# Econ 148 — Machine Learning for Economists
##  Part 2 Regularization: Ridge and Lasso
### Data: National Longitudinal Survey of Young Men (NLS) Panel, 1976–1982

**The question:** We want to predict log wages using occupation, industry, year, region, and personal characteristics — plus interactions between them. With 212 candidate features on 4,360 observations, how do we pick the right controls?

**Why this matters for economists:** This is the "kitchen sink" regression problem. You have occupation dummies, industry dummies, year fixed effects, and interactions between them. OLS will fit all 212 — but should it?

---

| Variable group | Variables | Count |
|---|---|---|
| Occupation dummies | `occ1`–`occ9` (manager, sales, clerical, craft, operative, transport, farm, service, professional) | 9 |
| Industry dummies | agriculture, business, construction, entertainment, finance, manufacturing, mining, personal svcs, professional, public admin, transport, trade | 12 |
| Year dummies | 1981–1987 | 7 |
| Region dummies | North Central, Northeast, South, Rural | 4 |
| Personal characteristics | black, Hispanic, education, experience, experience², married, union, hours, poor health | 9 |
| **Interactions** | occupation × industry, occupation × year | **171** |
| **Total** | | **212** |

In [1]:
try: import wooldridge
except ImportError:
    !pip install wooldridge -q
    import wooldridge

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [5]:
df_raw = wooldridge.data('wagepan')

print(f"lwage: mean={df_raw['lwage'].mean():.3f}, SD={df_raw['lwage'].std():.3f}, range=[{df_raw['lwage'].min():.2f}, {df_raw['lwage'].max():.2f}]")

lwage: mean=1.649, SD=0.533, range=[-3.58, 4.05]


In [4]:
wooldridge.data('wagepan', description=True)


name of dataset: wagepan
no of variables: 44
no of observations: 4360

+----------+------------------------+
| variable | label                  |
+----------+------------------------+
| nr       | person identifier      |
| year     | 1980 to 1987           |
| agric    | =1 if in agriculture   |
| black    | =1 if black            |
| bus      |                        |
| construc | =1 if in construction  |
| ent      |                        |
| exper    | labor mkt experience   |
| fin      |                        |
| hisp     | =1 if Hispanic         |
| poorhlth | =1 if in poor health   |
| hours    | annual hours worked    |
| manuf    | =1 if in manufacturing |
| married  | =1 if married          |
| min      |                        |
| nrthcen  | =1 if north central    |
| nrtheast | =1 if north east       |
| occ1     |                        |
| occ2     |                        |
| occ3     |                        |
| occ4     |                        |
| occ5     |     

---
## Step 1: Build the Feature Matrix

We start with 41 base features (occupation/industry/year/region dummies + personal characteristics), then add occupation × industry and occupation × year interactions to reach 212 features.

These interactions have economic content:
- **occ3 × manuf**: clerical workers in manufacturing — a specific labor market segment
- **occ1 × d85**: managers in 1985 — captures time-varying returns to management
- **occ6 × construc**: transport workers in construction — industry-specific occupation effects

An economist might want all of these. The question is which ones actually matter.

In [ ]:
# Define variable groups
occ_cols  = [f'occ{i}' for i in range(1, 10)]
ind_cols  = ['agric','bus','construc','ent','fin','manuf','min','per','pro','pub','tra','trad']
yr_cols   = ['d81','d82','d83','d84','d85','d86','d87']
reg_cols  = ['nrthcen','nrtheast','south','rur']
pers_cols = ['black','hisp','educ','exper','expersq','married','union','hours','poorhlth']

# Human-readable occupation labels
occ_labels = {
    'occ1': 'Manager/Admin',   'occ2': 'Sales',        'occ3': 'Clerical',
    'occ4': 'Craft/Repair',    'occ5': 'Operative',    'occ6': 'Transport',
    'occ7': 'Farm/Labor',      'occ8': 'Service',      'occ9': 'Professional'
}
ind_labels = {
    'agric': 'Agriculture', 'bus': 'Business Svcs', 'construc': 'Construction',
    'ent': 'Entertainment', 'fin': 'Finance',        'manuf': 'Manufacturing',
    'min': 'Mining',        'per': 'Personal Svcs',  'pro': 'Prof. Services',
    'pub': 'Public Admin',  'tra': 'Transport',      'trad': 'Trade'
}

In [ ]:
# Build feature matrix using pd.concat — avoids DataFrame fragmentation
base_df = df_raw[occ_cols + ind_cols + yr_cols + reg_cols + pers_cols].copy()

occ_x_ind_dict = {f'{o}_x_{i}': df_raw[o] * df_raw[i]
                  for o in occ_cols for i in ind_cols}
occ_x_yr_dict  = {f'{o}_x_{y}': df_raw[o] * df_raw[y]
                  for o in occ_cols for y in yr_cols}

features_df = pd.concat(
    [base_df,
     pd.DataFrame(occ_x_ind_dict, index=df_raw.index),
     pd.DataFrame(occ_x_yr_dict,  index=df_raw.index)],
    axis=1
)

feature_names = features_df.columns.tolist()
n_base  = len(occ_cols + ind_cols + yr_cols + reg_cols + pers_cols)
n_inter = len(occ_x_ind_dict) + len(occ_x_yr_dict)

print(f"Base features: {n_base}  |  Interactions: {n_inter} (occ×industry: {len(occ_x_ind_dict)}, occ×year: {len(occ_x_yr_dict)})")
print(f"Total p={len(feature_names)}, n={len(df_raw):,}, p/n={len(feature_names)/len(df_raw):.3f}")

In [ ]:
# Train/test split and standardization
X = features_df.values
y = df_raw['lwage'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Training: {len(X_train):,} obs  |  Test: {len(X_test):,} obs")

**Why standardize before regularization?** The penalty λ∑β² treats all coefficients equally — but without standardization it would penalize large-scale variables more than small-scale ones simply because of units (e.g., `hours` ≈ 2000/yr vs `black` ∈ {0,1}). Standardizing first ensures the penalty reflects each variable's contribution, not its scale.

---
## Step 2: The Problem with OLS — Too Many Controls

With 212 features and 4,360 observations, OLS can still technically fit the model. But watch what happens to the coefficient estimates.

In [ ]:
# OLS with all 212 features
ols = LinearRegression(fit_intercept=True)
ols.fit(X_train_s, y_train)

ols_train_r2 = r2_score(y_train, ols.predict(X_train_s))
ols_test_r2  = r2_score(y_test,  ols.predict(X_test_s))

print("OLS with all 212 features:")
print(f"  Train R²:              {ols_train_r2:.4f}")
print(f"  Test R²:               {ols_test_r2:.4f}")
print(f"  Coefficient range:     [{ols.coef_.min():.3f}, {ols.coef_.max():.3f}]")
print(f"  Coefficients > |0.5|:  {(np.abs(ols.coef_) > 0.5).sum()}")

The train–test R² gap (0.3449 → 0.2264) shows OLS is fitting noise in the 171 interaction terms. The model is technically identified (p/n ≈ 0.05), but many near-zero interactions inflate variance without reducing bias — this is the overfitting regularization is designed to address.

---
## Step 3: Regularization

Both Ridge and Lasso add a penalty to the OLS objective that shrinks coefficients toward zero:

$$\text{Ridge: } \min_\beta \|y - X\beta\|^2 + \lambda \sum_j \beta_j^2 \qquad \text{(L2 — shrinks but keeps all)}$$

$$\text{Lasso: } \min_\beta \|y - X\beta\|^2 + \lambda \sum_j |\beta_j| \qquad \text{(L1 — sets some to exactly zero)}$$

The key geometric reason Lasso produces zeros: the L1 "ball" has corners at the axes, so the optimum often lands exactly on a coordinate axis (coefficient = 0). Ridge's smooth L2 ball has no corners, so it rarely produces exact zeros.

### Choose λ by cross-validation

In [ ]:
# Cross-validate to find optimal λ for Lasso
print("Running 5-fold CV for Lasso... (this takes ~30 seconds)")
lasso_cv = LassoCV(cv=5, max_iter=20000, n_alphas=100, random_state=42)
lasso_cv.fit(X_train_s, y_train)
best_lasso_alpha = lasso_cv.alpha_

# Cross-validate for Ridge
alphas_ridge = np.logspace(-2, 4, 100)
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5)
ridge_cv.fit(X_train_s, y_train)
best_ridge_alpha = ridge_cv.alpha_

print(f"\nCV-selected λ for Lasso: {best_lasso_alpha:.4f}")
print(f"CV-selected λ for Ridge: {best_ridge_alpha:.2f}")

Running 5-fold CV for Lasso... (this takes ~30 seconds)

CV-selected λ for Lasso: 0.0104
CV-selected λ for Ridge: 811.13


In [ ]:
# Fit final models at CV-selected λ
final_lasso = Lasso(alpha=best_lasso_alpha, max_iter=20000)
final_ridge  = Ridge(alpha=best_ridge_alpha)
final_lasso.fit(X_train_s, y_train)
final_ridge.fit(X_train_s, y_train)

lasso_nz = (final_lasso.coef_ != 0).sum()

# Summary table
print("=" * 65)
print(f"{'Model':<28} {'Train R²':>9} {'Test R²':>9} {'# Features':>12}")
print("=" * 65)
models = {
    'OLS (all 212 features)':          (ols, X_train_s, X_test_s),
    f'Ridge (λ={best_ridge_alpha:.1f})': (final_ridge, X_train_s, X_test_s),
    f'Lasso (λ={best_lasso_alpha:.4f})': (final_lasso, X_train_s, X_test_s),
}
for name, (model, Xtr, Xte) in models.items():
    tr = r2_score(y_train, model.predict(Xtr))
    te = r2_score(y_test,  model.predict(Xte))
    nf = (model.coef_ != 0).sum()
    print(f"{name:<28} {tr:>9.4f} {te:>9.4f} {nf:>12}")
print("=" * 65)

---
## Step 4: Visualizing the Difference — Regularization Paths

As λ increases (more regularization), what happens to the coefficients?

In [ ]:
# Compute regularization paths
lambdas = np.logspace(-4, 0.5, 80)
lasso_paths, ridge_paths = [], []

for lam in lambdas:
    l = Lasso(alpha=lam, max_iter=10000).fit(X_train_s, y_train)
    r = Ridge(alpha=lam * 1000).fit(X_train_s, y_train)
    lasso_paths.append(l.coef_)
    ridge_paths.append(r.coef_)

lasso_paths = np.array(lasso_paths)
ridge_paths = np.array(ridge_paths)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cmap = plt.cm.tab20
log_lam = np.log10(lambdas)

show_base  = list(range(n_base))
show_inter = list(range(n_base, n_base + n_inter, 8))
show_idx   = show_base + show_inter

# Lasso path
ax = axes[0]
for j in show_idx:
    col = cmap((j % 20) / 20)
    ax.plot(log_lam, lasso_paths[:, j], color=col, lw=1.2, alpha=0.7)
ax.axvline(np.log10(best_lasso_alpha), color='black', lw=2, linestyle='--',
           label=f'CV-selected λ = {best_lasso_alpha:.4f}')
ax.axhline(0, color='gray', lw=0.6)
ax.set_xlabel('log₁₀(λ)  →  more regularization →')
ax.set_ylabel('Standardized Coefficient')
ax.set_title('Lasso (L1) Paths\nCoefficients hit EXACTLY zero', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
n_zero_at_cv = (lasso_paths[np.argmin(np.abs(lambdas - best_lasso_alpha)), :] == 0).sum()
ax.annotate(f'{n_zero_at_cv} features\nzeroed out →',
            xy=(np.log10(best_lasso_alpha), 0.02),
            xytext=(np.log10(best_lasso_alpha) - 0.8, 0.12),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9)

# Ridge path
ax = axes[1]
ridge_lambdas_plot = np.logspace(-2, 4, 80)
ridge_paths2 = np.array([Ridge(alpha=a).fit(X_train_s, y_train).coef_
                          for a in ridge_lambdas_plot])
for j in show_idx:
    col = cmap((j % 20) / 20)
    ax.plot(np.log10(ridge_lambdas_plot), ridge_paths2[:, j], color=col, lw=1.2, alpha=0.7)
ax.axvline(np.log10(best_ridge_alpha), color='black', lw=2, linestyle='--',
           label=f'CV-selected λ = {best_ridge_alpha:.1f}')
ax.axhline(0, color='gray', lw=0.6)
ax.set_xlabel('log₁₀(λ)  →  more regularization →')
ax.set_ylabel('Standardized Coefficient')
ax.set_title('Ridge (L2) Paths\nCoefficients shrink but NEVER reach zero', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Figure 1: Regularization Paths — NLS Wage Panel\n'
             '(41 base features + sample of 171 occupation×industry/year interactions)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('reg_paths.png', bbox_inches='tight', dpi=130)
plt.show()

---
## Step 5: What Does Lasso Actually Select?

The economic punchline: which features survive Lasso's selection?

In [ ]:
# Separate selected vs dropped features
coef_df = pd.DataFrame({
    'feature':    feature_names,
    'ols_coef':   ols.coef_,
    'lasso_coef': final_lasso.coef_,
    'ridge_coef': final_ridge.coef_,
})
coef_df['selected'] = coef_df['lasso_coef'] != 0
coef_df['is_base']  = coef_df['feature'].isin(occ_cols + ind_cols + yr_cols + reg_cols + pers_cols)
coef_df['abs_lasso'] = coef_df['lasso_coef'].abs()

selected  = coef_df[coef_df['selected']].sort_values('abs_lasso', ascending=False)
dropped   = coef_df[~coef_df['selected']]

n_base_sel   = selected['is_base'].sum()
n_inter_sel  = (~selected['is_base']).sum()
n_base_drop  = (~coef_df[coef_df['is_base']]['selected']).sum()
n_inter_drop = (~coef_df[~coef_df['is_base']]['selected']).sum()

In [ ]:
print(f"Lasso selected {len(selected)} / {len(feature_names)} features")
print(f"  Base features:        {n_base_sel} / {n_base} selected, {n_base_drop} dropped")
print(f"  Interaction features: {n_inter_sel} / {n_inter} selected, {n_inter_drop} dropped")
print()
print("Top 15 features by |coefficient|:")
print(selected[['feature','lasso_coef','ols_coef']].head(15).to_string(index=False))
print()
dropped_base = dropped[dropped['is_base']]['feature'].tolist()
print("Base features dropped:", ", ".join(dropped_base) if dropped_base else "none")

In [ ]:
# ── Figure 2: Lasso vs Ridge coefficients for BASE features only ──────────────
base_df_plot = coef_df[coef_df['is_base']].copy()
base_df_plot = base_df_plot.sort_values('lasso_coef')

label_map = {**occ_labels, **ind_labels,
             **{f'd{y}': str(y) for y in range(81,88)},
             'nrthcen': 'N. Central', 'nrtheast': 'Northeast',
             'south': 'South', 'rur': 'Rural',
             'black': 'Black', 'hisp': 'Hispanic', 'educ': 'Education',
             'exper': 'Experience', 'expersq': 'Experience²',
             'married': 'Married', 'union': 'Union',
             'hours': 'Hours/Wk', 'poorhlth': 'Poor Health'}
base_df_plot['label'] = base_df_plot['feature'].map(lambda x: label_map.get(x, x))

fig, ax = plt.subplots(figsize=(11, 10))
x = np.arange(len(base_df_plot))
width = 0.38

colors_lasso = ['steelblue' if v != 0 else 'lightgray' for v in base_df_plot['lasso_coef']]

ax.barh(x - width/2, base_df_plot['lasso_coef'], width,
        color=colors_lasso, edgecolor='white', label='Lasso')
ax.barh(x + width/2, base_df_plot['ridge_coef'], width,
        color='#f0a060', alpha=0.8, edgecolor='white', label='Ridge')

ax.set_yticks(x)
ax.set_yticklabels(base_df_plot['label'], fontsize=8.5)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Standardized Coefficient')
ax.set_title(
    'Figure 2: Lasso vs Ridge — Base Feature Coefficients\n'
    f'(NLS Panel, λ chosen by 5-fold CV | gray = zeroed by Lasso)',
    fontsize=12, fontweight='bold'
)

lasso_patch = mpatches.Patch(color='steelblue', label=f'Lasso (λ={best_lasso_alpha:.4f})')
zero_patch  = mpatches.Patch(color='lightgray', label='Dropped by Lasso (= 0)')
ridge_patch = mpatches.Patch(color='#f0a060',   label=f'Ridge (λ={best_ridge_alpha:.1f})')
ax.legend(handles=[lasso_patch, zero_patch, ridge_patch], fontsize=10, loc='lower right')

plt.tight_layout()
plt.savefig('lasso_vs_ridge_base.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── Figure 3: Interaction sparsity — the real selection story ─────────────────
coef_sorted = coef_df.sort_values('lasso_coef')
x_all = np.arange(len(coef_sorted))

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Top: Lasso — sparse
ax = axes[0]
colors_all = ['steelblue' if s else 'lightgray' for s in coef_sorted['selected']]
ax.bar(x_all, coef_sorted['lasso_coef'], color=colors_all, width=1.0, edgecolor='none')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Coefficient')
ax.set_title(
    f'Lasso: {lasso_nz} / {len(feature_names)} features selected  '
    f'({len(feature_names)-lasso_nz} zeroed out)',
    fontsize=11, fontweight='bold'
)
ax.set_xticks([])
lasso_p = mpatches.Patch(color='steelblue', label='Selected (non-zero)')
zero_p  = mpatches.Patch(color='lightgray', label='Dropped (= 0)')
ax.legend(handles=[lasso_p, zero_p], fontsize=9)
ax.axvline(n_base, color='tomato', lw=1.5, linestyle=':', alpha=0.7)
ax.annotate('← Base features | Interaction features →',
            xy=(n_base, ax.get_ylim()[1]*0.85),
            fontsize=8, color='tomato', ha='center')

# Bottom: Ridge — dense
ax = axes[1]
ridge_sorted = coef_df.sort_values('lasso_coef')
ax.bar(x_all, ridge_sorted['ridge_coef'], color='#f0a060', width=1.0, edgecolor='none', alpha=0.8)
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Coefficient')
ax.set_title(
    f'Ridge: all {len(feature_names)} features retained (shrunk but non-zero)',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Features (sorted by Lasso coefficient value)')
ax.set_xticks([])
ax.axvline(n_base, color='tomato', lw=1.5, linestyle=':', alpha=0.7)

plt.suptitle(
    'Figure 3: Sparsity — Lasso vs Ridge across all 212 Features\n'
    '(NLS Wage Panel: 9 occupations × 12 industries × 7 years + personal characteristics)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('sparsity_comparison.png', bbox_inches='tight', dpi=130)
plt.show()

---
## Step 6: Why Does This Matter for Econometrics?

The variable selection Lasso performs is not just a prediction convenience — it has direct implications for causal inference.

In [ ]:
# Union wage premium under three control specifications
union_idx = feature_names.index('union')
basic_idx = [feature_names.index(f) for f in
             ['educ','exper','expersq','black','hisp','married','union']]
lasso_idx = [i for i, f in enumerate(feature_names) if final_lasso.coef_[i] != 0]
all_idx   = list(range(len(feature_names)))

def ols_union_premium(X_s, y, col_indices, union_col_idx):
    """Run OLS and return (coef, SE, CI_lo, CI_hi) for the union variable."""
    X_sub = sm.add_constant(X_s[:, col_indices])
    res   = sm.OLS(y, X_sub).fit()
    if union_col_idx in col_indices:
        pos = col_indices.index(union_col_idx) + 1
        b, se = res.params[pos], res.bse[pos]
        ci = res.conf_int()[pos]
        return b, se, ci[0], ci[1]
    return None

specs = [
    ('Naive OLS (7 controls)',                    basic_idx),
    ('Kitchen-sink OLS (212 features)',            all_idx),
    (f'Lasso-selected OLS ({len(lasso_idx)} features)', lasso_idx),
]

In [ ]:
print("Union Wage Premium Estimates (training data, standardized features)")
print("=" * 70)
print(f"{'Specification':<38} {'Coef':>7} {'SE':>7} {'95% CI'}")
print("=" * 70)
for name, idx in specs:
    result = ols_union_premium(X_train_s, y_train, idx, union_idx)
    if result:
        b, se, lo, hi = result
        print(f"{name:<38} {b:>7.4f} {se:>7.4f}  [{lo:.4f}, {hi:.4f}]")
    else:
        print(f"{name:<38} union not in this spec")
print("=" * 70)

The kitchen-sink OLS inflates standard errors by fitting all 212 noisy interactions. Lasso-selected controls recover a similar union premium estimate but don't beat OLS.

This is the intuition behind Belloni, Chernozhukov & Hansen (2014) "double Lasso": use Lasso to select controls, then run OLS for inference. We'll formalize this in the next part of the lecture.

---
## Summary

### What we did
Used the NLS wage panel to build a 212-feature regression problem — 9 occupations, 12 industries, 7 year FEs, region and personal controls, plus all occupation×industry and occupation×year interactions. Then compared OLS, Ridge, and Lasso.

### What we learned

| | OLS | Ridge | Lasso |
|---|---|---|---|
| Objective | Minimize residuals | Minimize residuals + L2 penalty | Minimize residuals + L1 penalty |
| Coefficients | Unbiased, noisy | Shrunk, non-zero | Shrunk, many exactly zero |
| Variable selection | None | None | **Yes — automatic** |
| Test R² | OK | OK | **Slightly better** |
| Use case | Causal inference (low p) | Prediction with correlated features | Prediction + control selection |

### The bridge to causal inference

> **Key insight:** Lasso selects which occupation × industry controls to include before you run your causal regression. This is exactly what Belloni, Chernozhukov & Hansen (2014) — "double Lasso" — formalize. You run Lasso twice: once regressing the outcome on controls, once regressing the treatment on controls. Include all variables selected by either. Then run OLS on the reduced set.
>
> The result: better control selection than "kitchen sink" OLS, tighter standard errors, and — crucially — valid inference even when p is large.

---
**Data:** Vella & Verbeek (1998), "Whose Wages Do Unions Raise?" NLS Young Men panel, via Wooldridge (2012).  
**References:** Belloni, Chernozhukov & Hansen (2014). *Journal of Economic Perspectives*.  Tibshirani (1996). "Regression shrinkage and selection via the Lasso." *JRSS-B*.